# VWAP Statistical Model Research

## Phase 2 — Statistical Research Framework

### Research objectives

- Establish a reproducible historical research environment.
- Validate historical dataset quality and provenance.
- Reconstruct causal VWAP and probability-band observations.
- Reproduce the existing solid RMS and Gaussian ghost baselines.
- Evaluate statistical calibration and predictive performance.
- Establish chronological validation and out-of-sample protocols.

### Research principles

The existing engine is treated as a reference baseline rather
than an assumed optimal model.

Historical datasets must remain unchanged. Timestamp
conventions, overlapping observations, missing candles and
mixed-frequency data must be investigated before modelling.

Candidate distributions and trading rules must be evaluated
without look-ahead bias or retrospective parameter selection.

## 1. Research environment and dataset inventory

In [ ]:

from pathlib import Path
import sys

import pandas as pd
import numpy as np


# Locate the repository root whether the notebook starts
# from the project directory or the notebooks directory.
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "research_config.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.research_config import RESEARCH_CONFIG


DATA_DIR = RESEARCH_CONFIG["data_dir"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Python interpreter: {sys.executable}")
print(f"Instrument: {RESEARCH_CONFIG['instrument']}")
print(f"Timeframe: {RESEARCH_CONFIG['timeframe']}")
print(f"Baseline: {RESEARCH_CONFIG['baseline_dataset']}")


In [ ]:

dataset_paths = sorted(DATA_DIR.glob("*.csv"))

assert dataset_paths, "No historical CSV datasets found."

dataset_inventory = pd.DataFrame(
    [
        {
            "filename": path.name,
            "size_mb": round(path.stat().st_size / 1024**2, 2),
            "columns": tuple(pd.read_csv(path, nrows=0).columns),
            "is_baseline": (
                path.name == RESEARCH_CONFIG["baseline_dataset"]
            ),
        }
        for path in dataset_paths
    ]
)

assert dataset_inventory["is_baseline"].sum() == 1, (
    "Expected exactly one baseline dataset."
)

print(f"Historical datasets: {len(dataset_inventory)}")
print(f"Total size: {dataset_inventory['size_mb'].sum():.2f} MB")

display(dataset_inventory)


## 2. Historical Dataset Registry and Provenance

Historical observations are separated into three categories:

- Legacy historical datasets retained for baseline reproduction
  and comparison.
- Immutable raw MT5 acquisitions with recorded source metadata
  and content-derived dataset identities.
- Validated research datasets produced only after independent
  timestamp and data-quality assessment.

External storage allows datasets to be shared across research
branches without duplication or inclusion in Git history.

Successful file-integrity verification does not establish
historical timezone accuracy, trading-session completeness
or eligibility for statistical modelling.

In [ ]:
from src.research_data_diagnostics import sha256_file

legacy_dataset_registry = pd.DataFrame(
    [
        {
            "filename": path.name,
            "dataset_type": "legacy_historical",
            "dataset_id": f"sha256:{sha256_file(path)}",
            "file_size_bytes": path.stat().st_size,
            "is_baseline": (
                path.name == RESEARCH_CONFIG["baseline_dataset"]
            ),
            "provenance_status": "not_independently_verified",
            "research_eligibility": "pending_validation",
        }
        for path in dataset_paths
    ]
)

display(legacy_dataset_registry)

In [ ]:
from src.research_dataset_registry import (
    discover_raw_datasets,
)

RAW_DATA_DIR = RESEARCH_CONFIG["raw_data_dir"]
VALIDATED_DATA_DIR = RESEARCH_CONFIG["validated_data_dir"]

print(f"External raw data: {RAW_DATA_DIR}")
print(f"Validated data: {VALIDATED_DATA_DIR}")

raw_dataset_records = discover_raw_datasets(
    RAW_DATA_DIR
)

raw_dataset_registry = pd.DataFrame(
    raw_dataset_records
)

print(
    f"Registered raw acquisitions: "
    f"{len(raw_dataset_registry)}"
)

if not raw_dataset_registry.empty:
    display(
        raw_dataset_registry[
            [
                "dataset_label",
                "symbol",
                "timeframe",
                "rows",
                "broker_server",
                "integrity_verified",
                "timestamp_alignment",
                "research_eligibility",
            ]
        ]
    )
else:
    print("No raw acquisitions registered.")

In [ ]:
if not raw_dataset_registry.empty:
    display(
        raw_dataset_registry[
            [
                "dataset_label",
                "dataset_id",
                "requested_start",
                "requested_end_exclusive",
                "observed_first_timestamp",
                "observed_last_timestamp",
                "acquisition_started_utc",
                "code_version",
                "working_tree_dirty",
            ]
        ]
    )

## 3. Historical Timestamp and Data Quality Investigation

Raw MT5 timestamps must not be assumed to represent genuine
UTC solely because they have been decoded as Unix timestamps.

This investigation evaluates:

- Historical trading coverage and intraday observation gaps.
- Broker-server timestamp conventions.
- Historical daylight-saving transitions.
- Agreement between newly acquired and legacy observations.

Historical broker announcements provide independent evidence
against which timestamp interpretations can be assessed.

Raw observations remain unchanged throughout this investigation.

In [ ]:
annual_coverage_records = []
daily_coverage_records = []

for record in raw_dataset_records:

    frame = pd.read_csv(
        record["csv_path"],
        usecols=["mt5_time_raw"],
    )

    timestamps = pd.to_datetime(
        frame["mt5_time_raw"],
        unit="s",
        utc=True,
    )

    # These are provisional raw clock labels, not verified UTC.
    year = pd.Timestamp(record["requested_start"]).year

    daily = timestamps.groupby(
        timestamps.dt.normalize()
    ).agg(
        first="min",
        last="max",
        rows="size",
        unique_minutes="nunique",
    )

    daily["span_minutes"] = (
        (daily["last"] - daily["first"])
        .dt.total_seconds() / 60
    ).astype(int) + 1

    daily["internal_unobserved_minutes"] = (
        daily["span_minutes"] - daily["unique_minutes"]
    )

    daily["year"] = year
    daily["month"] = daily.index.month
    daily["raw_date"] = daily.index.strftime("%Y-%m-%d")

    daily_coverage_records.append(
        daily.reset_index(drop=True)
    )

    annual_coverage_records.append({
        "year": year,
        "rows": len(frame),
        "first_raw_timestamp": timestamps.min(),
        "last_raw_timestamp": timestamps.max(),
        "observed_days": len(daily),
        "typical_daily_rows": daily["rows"].mode().iloc[0],
        "days_with_internal_gaps": (
            daily["internal_unobserved_minutes"] > 0
        ).sum(),
        "internal_unobserved_minutes": (
            daily["internal_unobserved_minutes"].sum()
        ),
        "most_common_open": (
            daily["first"].dt.strftime("%H:%M").mode().iloc[0]
        ),
        "most_common_last_candle": (
            daily["last"].dt.strftime("%H:%M").mode().iloc[0]
        ),
    })

    del frame, timestamps

annual_coverage = pd.DataFrame(
    annual_coverage_records
).sort_values("year")

daily_coverage = pd.concat(
    daily_coverage_records,
    ignore_index=True,
)

monthly_coverage = daily_coverage.pivot_table(
    index="month",
    columns="year",
    values="rows",
    aggfunc="sum",
).reindex(range(1, 13))

print("ANNUAL HISTORICAL COVERAGE")
display(annual_coverage)

print("MONTHLY CANDLE COUNTS")
display(monthly_coverage)

In [ ]:
# Independent historical FTMO trading announcements.
# Format: date, announced server close, GMT offset, source.

timestamp_references = [
    (
        "2023-05-29", "20:00", 3,
        "https://ftmo.com/en/blog/trading-updates/trading-update-25-may-2023/",
    ),
    (
        "2023-11-23", "20:00", 2,
        "https://ftmo.com/en/blog/trading-updates/trading-update-23-nov-2023/",
    ),
    (
        "2024-09-02", "20:00", 3,
        "https://ftmo.com/en/blog/trading-updates/trading-update-29-aug-2024/",
    ),
    (
        "2024-11-28", "20:00", 2,
        "https://ftmo.com/en/blog/trading-updates/trading-update-21-nov-2024/",
    ),
    (
        "2025-01-09", "16:30", 2,
        "https://ftmo.com/en/blog/trading-updates/trading-update-9-jan-2025/",
    ),
    (
        "2025-06-19", "20:00", 3,
        "https://ftmo.com/en/blog/trading-updates/trading-update-5-jun-2025/",
    ),
    (
        "2025-07-04", "20:15", 3,
        "https://ftmo.com/en/blog/trading-updates/trading-update-3-jul-2025/",
    ),
    (
        "2026-01-19", "20:00", 2,
        "https://ftmo.com/en/blog/trading-updates/trading-update-15-jan-2026/",
    ),
]

timestamp_evidence_records = []

for date, close, offset, source in timestamp_references:

    observations = daily_coverage.loc[
        daily_coverage["raw_date"].eq(date)
    ]

    # A candle is labelled by its opening time.
    expected_server_last = (
        pd.Timestamp(f"{date} {close}", tz="UTC")
        - pd.Timedelta(minutes=1)
    )

    expected_utc_last = (
        expected_server_last
        - pd.Timedelta(hours=offset)
    )

    if len(observations) != 1:
        observed_last = pd.NaT
        server_error = np.nan
        utc_error = np.nan
        interpretation = "unavailable_or_ambiguous"

    else:
        observed_last = observations["last"].iloc[0]

        server_error = (
            observed_last - expected_server_last
        ).total_seconds() / 60

        utc_error = (
            observed_last - expected_utc_last
        ).total_seconds() / 60

        if server_error == 0:
            interpretation = "exact_server_clock_match"

        elif utc_error == 0:
            interpretation = "exact_utc_match"

        else:
            interpretation = "no_exact_match"

    timestamp_evidence_records.append({
        "date": date,
        "broker_offset_hours": offset,
        "announced_server_close": close,
        "expected_server_last": (
            expected_server_last.strftime("%H:%M")
        ),
        "expected_utc_last": (
            expected_utc_last.strftime("%H:%M")
        ),
        "observed_raw_last": (
            observed_last.strftime("%H:%M")
            if pd.notna(observed_last) else None
        ),
        "server_error_minutes": server_error,
        "utc_error_minutes": utc_error,
        "interpretation": interpretation,
        "source": source,
    })

timestamp_evidence = pd.DataFrame(
    timestamp_evidence_records
)

display(
    timestamp_evidence.drop(columns="source")
)

In [ ]:
# FTMO's documented US daylight-saving transition dates.
# Applicable to our US100.cash weekday observations.

from src.research_data_diagnostics import (
    FTMO_US100_DST_DATES,
    ftmo_us100_utc_from_raw,
)

ftmo_dst_dates = FTMO_US100_DST_DATES

ftmo_ohlc_columns = ["open", "high", "low", "close"]

comparison_columns = ftmo_ohlc_columns + [
    "tick_volume",
    "spread",
    "real_volume",
]

raw_frames = []
clock_records = []

for record in raw_dataset_records:

    year = pd.Timestamp(record["requested_start"]).year

    # Retain 2021 externally, but exclude it from primary research.
    if year not in ftmo_dst_dates:
        continue

    frame = pd.read_csv(
        record["csv_path"],
        usecols=["mt5_time_raw"] + comparison_columns,
    )

    raw_clock = pd.to_datetime(
        frame["mt5_time_raw"],
        unit="s",
        utc=True,
    )

    candidate_utc = ftmo_us100_utc_from_raw(
        frame["mt5_time_raw"]
    )

    offset_hours = (
        raw_clock - candidate_utc
    ).dt.total_seconds() / 3600

    summer = offset_hours.eq(3)

    candidate_ny = candidate_utc.dt.tz_convert(
        RESEARCH_CONFIG["session_timezone"]
    )

    frame["raw_clock"] = raw_clock
    frame["candidate_utc"] = candidate_utc

    clock_records.append({
        "year": year,
        "winter_observations": int((~summer).sum()),
        "summer_observations": int(summer.sum()),
        "first_candidate_utc": candidate_utc.min(),
        "last_candidate_utc": candidate_utc.max(),
        "first_candidate_ny": candidate_ny.min(),
        "last_candidate_ny": candidate_ny.max(),
        "candidate_duplicate_timestamps": int(
            candidate_utc.duplicated().sum()
        ),
        "candidate_order_preserved": (
            candidate_utc.is_monotonic_increasing
        ),
    })

    raw_frames.append(frame)

clock_diagnostics = pd.DataFrame(clock_records)

display(clock_diagnostics)

raw_comparison_data = pd.concat(
    raw_frames,
    ignore_index=True,
)

del raw_frames

assert raw_comparison_data["raw_clock"].is_unique
assert raw_comparison_data["candidate_utc"].is_unique

# Additional independently documented 2022 holiday check.
thanksgiving_2022 = daily_coverage.loc[
    daily_coverage["raw_date"].isin([
        "2022-11-24",
        "2022-11-25",
    ]),
    ["raw_date", "first", "last", "rows"],
]

print("2022 Thanksgiving — announced server close: 20:00")
display(thanksgiving_2022)

In [ ]:
comparison_records = []

for path in dataset_paths:

    if "2021" in path.name:
        continue

    legacy = pd.read_csv(
        path,
    )

    legacy["datetime"] = pd.to_datetime(
        legacy["datetime"],
        utc=True,
        errors="raise",
    )

    assert legacy["datetime"].is_unique, (
        f"Duplicate timestamps in {path.name}"
    )

    shared_columns = [
        column
        for column in comparison_columns
        if column in legacy.columns
    ]

    assert set(ftmo_ohlc_columns).issubset(shared_columns)

    for interpretation in ["raw_clock", "candidate_utc"]:

        reference = raw_comparison_data[
            [interpretation] + shared_columns
        ].rename(
            columns={interpretation: "datetime"}
        )

        matched = legacy[
            ["datetime"] + shared_columns
        ].merge(
            reference,
            on="datetime",
            how="inner",
            suffixes=("_legacy", "_new"),
            validate="one_to_one",
        )

        result = {
            "dataset": path.name,
            "clock_interpretation": interpretation,
            "legacy_rows": len(legacy),
            "matched_rows": len(matched),
            "timestamp_overlap_pct": (
                100 * len(matched) / len(legacy)
            ),
        }

        if matched.empty:
            comparison_records.append(result)
            continue

        legacy_ohlc = matched[
            [
                f"{column}_legacy"
                for column in ftmo_ohlc_columns
            ]
        ].to_numpy(dtype=float)

        new_ohlc = matched[
            [
                f"{column}_new"
                for column in ftmo_ohlc_columns
            ]
        ].to_numpy(dtype=float)

        # Tolerance only accommodates floating-point noise.
        ohlc_agreement = np.isclose(
            legacy_ohlc,
            new_ohlc,
            rtol=0,
            atol=1e-8,
        ).all(axis=1)

        close_error = (
            matched["close_legacy"]
            - matched["close_new"]
        ).abs()

        result.update({
            "ohlc_matching_rows": int(
                ohlc_agreement.sum()
            ),
            "ohlc_agreement_pct": (
                100 * ohlc_agreement.mean()
            ),
            "ohlc_conflicting_rows": int(
                (~ohlc_agreement).sum()
            ),
            "median_abs_close_difference": (
                close_error.median()
            ),
            "p95_abs_close_difference": (
                close_error.quantile(0.95)
            ),
            "max_abs_close_difference": (
                close_error.max()
            ),
        })

        for column in [
            "tick_volume",
            "spread",
            "real_volume",
        ]:

            if column in shared_columns:

                agreement = matched[
                    f"{column}_legacy"
                ].eq(
                    matched[f"{column}_new"]
                )

                result[f"{column}_agreement_pct"] = (
                    100 * agreement.mean()
                )

        comparison_records.append(result)

legacy_clock_comparison = pd.DataFrame(
    comparison_records
)

print("LEGACY TIMESTAMP AND PRICE COMPARISON")

display(
    legacy_clock_comparison[
        [
            "dataset",
            "clock_interpretation",
            "legacy_rows",
            "matched_rows",
            "timestamp_overlap_pct",
            "ohlc_agreement_pct",
            "ohlc_conflicting_rows",
            "median_abs_close_difference",
            "p95_abs_close_difference",
        ]
    ].round(4)
)

print("2025 BASELINE — DETAILED COMPARISON")

display(
    legacy_clock_comparison.loc[
        legacy_clock_comparison["dataset"].eq(
            RESEARCH_CONFIG["baseline_dataset"]
        )
    ].T
)

In [ ]:
import json
from tempfile import TemporaryDirectory

from src.research_data_diagnostics import (
    diagnose_raw_candles,
    sha256_file,
)

VALIDATED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

validation_records = []

for record in raw_dataset_records:

    year = pd.Timestamp(
        record["requested_start"]
    ).year

    if year not in FTMO_US100_DST_DATES:
        continue

    if (
        record["symbol"] != "US100.cash"
        or record["broker_server"] != "FTMO-Server"
    ):
        raise ValueError("Unsupported acquisition source.")

    # Confirm the original acquisition remains unchanged.
    if sha256_file(record["csv_path"]) != record["sha256"]:
        raise ValueError("Raw dataset checksum mismatch.")

    # Preserve the original candle values as CSV text.
    frame = pd.read_csv(
        record["csv_path"],
        dtype=str,
        keep_default_na=False,
    )

    quality = diagnose_raw_candles(frame)

    integrity_checks = [
        "invalid_timestamp_rows",
        "non_minute_aligned_rows",
        "timestamp_mismatch_rows",
        "backward_timestamp_moves",
        "exact_duplicate_rows",
        "conflicting_timestamps",
        "invalid_ohlc_rows",
        "invalid_volume_spread_rows",
        "zero_tick_volume_rows",
    ]

    failures = {
        key: quality[key]
        for key in integrity_checks
        if quality[key] != 0
    }

    if failures:
        raise ValueError(f"{year}: {failures}")

    corrected = ftmo_us100_utc_from_raw(
        frame["mt5_time_raw"].astype("int64")
    )

    # Confirm every corrected candle had closed
    # before its original acquisition began.
    acquisition_start = pd.Timestamp(
        record["acquisition_started_utc"]
    )

    if (
        corrected + pd.Timedelta(minutes=1)
        > acquisition_start
    ).any():
        raise ValueError(
            f"Incomplete candle detected in {year}"
        )

    # Retain the original clock labels while making
    # datetime the corrected UTC research timestamp.
    frame = frame.rename(
        columns={"datetime": "raw_datetime"}
    )

    frame.insert(0, "datetime", corrected)

    label = record["dataset_label"].replace(
        "_raw_",
        "_utc_",
        1,
    )

    output_folder = VALIDATED_DATA_DIR / label
    csv_path = output_folder / f"{label}.csv"
    metadata_path = output_folder / f"{label}.json"

    provenance = {
        "dataset_label": label,
        "source_dataset_id": record["dataset_id"],
        "source_sha256": record["sha256"],
        "source_metadata_sha256": sha256_file(
            record["metadata_path"]
        ),
        "correction_source_sha256": sha256_file(
            PROJECT_ROOT / "src"
            / "research_data_diagnostics.py"
        ),
        "symbol": record["symbol"],
        "timeframe": record["timeframe"],
        "broker_server": record["broker_server"],
        "rows": len(frame),
        "first_utc": corrected.min().isoformat(),
        "last_utc": corrected.max().isoformat(),
        "timestamp_policy": "FTMO US DST GMT+2/GMT+3",
        "dst_transition_dates": list(
            FTMO_US100_DST_DATES[year]
        ),
        "timestamp_correction_applied": True,
        "raw_timestamps_preserved": True,
        "observations_removed": 0,
        "structural_integrity_passed": True,
        "session_coverage_verified": False,
        "research_eligibility": "pending_session_validation",
    }

    # Allow the notebook to be rerun without overwriting
    # an existing corrected dataset.
    if output_folder.exists():

        existing = json.loads(
            metadata_path.read_text(encoding="utf-8")
        )

        if (
            any(
                existing.get(key) != value
                for key, value in provenance.items()
            )
            or sha256_file(csv_path) != existing["sha256"]
        ):
            raise ValueError(
                f"Existing dataset requires review: {label}"
            )

        status = "already_verified"

    else:

        # Stage both files before publishing the directory.
        with TemporaryDirectory(
            dir=VALIDATED_DATA_DIR
        ) as temporary:

            staging = Path(temporary) / label
            staging.mkdir()

            staged_csv = staging / csv_path.name

            frame.to_csv(
                staged_csv,
                index=False,
            )

            provenance["sha256"] = sha256_file(
                staged_csv
            )

            provenance["dataset_id"] = (
                f"sha256:{provenance['sha256']}"
            )

            (staging / metadata_path.name).write_text(
                json.dumps(provenance, indent=2),
                encoding="utf-8",
            )

            staging.rename(output_folder)

        status = "exported"

    validation_records.append({
        "year": year,
        "dataset": label,
        "rows": len(frame),
        "first_utc": corrected.min(),
        "last_utc": corrected.max(),
        "status": status,
        "research_eligibility": (
            "pending_session_validation"
        ),
    })

    del frame

validated_dataset_inventory = pd.DataFrame(
    validation_records
)

display(validated_dataset_inventory)

## 4. Model-Independent Historical Coverage

Historical coverage is assessed independently of model
parameters and trading hypotheses.

All corrected observations from 2022–2026 are retained.

Daily coverage describes available observations and internal
timestamp gaps without assuming that every unobserved minute
represents a data error.

Research-window completeness is evaluated dynamically using
the requirements of each candidate model.

The original VWAP anchor, warmup, volatility estimators and
Gaussian parameters are retained exclusively as a reference
benchmark.

Chronological research partitions are established before
candidate model development.

In [ ]:
from src.research_sessions import (
    validate_timestamp_index,
    summarise_daily_coverage,
    assess_windows,
)

timestamp_parts = []
source_dates = []

for record in sorted(
    raw_dataset_records,
    key=lambda r: r["requested_start"],
):

    year = pd.Timestamp(
        record["requested_start"]
    ).year

    if year not in FTMO_US100_DST_DATES:
        continue

    label = record["dataset_label"].replace(
        "_raw_", "_utc_", 1
    )

    folder = VALIDATED_DATA_DIR / label

    csv_path = folder / f"{label}.csv"
    metadata_path = folder / f"{label}.json"

    metadata = json.loads(
        metadata_path.read_text(encoding="utf-8")
    )

    if metadata["source_dataset_id"] != record["dataset_id"]:
        raise ValueError(
            f"Incorrect source provenance: {label}"
        )

    if not metadata["structural_integrity_passed"]:
        raise ValueError(
            f"Structural validation failed: {label}"
        )

    if sha256_file(csv_path) != metadata["sha256"]:
        raise ValueError(
            f"Corrected dataset checksum mismatch: {label}"
        )

    timestamps = pd.read_csv(
        csv_path,
        usecols=["datetime"],
    )["datetime"]

    timestamps = validate_timestamp_index(
        pd.to_datetime(
            timestamps,
            utc=True,
            errors="raise",
        )
    )

    if len(timestamps) != metadata["rows"]:
        raise ValueError(
            f"Row count mismatch: {label}"
        )

    timestamp_parts.append(timestamps)

    dates = timestamps.tz_convert(
        RESEARCH_CONFIG["session_timezone"]
    ).strftime("%Y-%m-%d").unique()

    source_dates.append(
        pd.DataFrame({
            "date": dates,
            "dataset_id": metadata["dataset_id"],
        })
    )

all_research_timestamps = validate_timestamp_index(
    timestamp_parts[0].append(timestamp_parts[1:])
)

local_timestamps = all_research_timestamps.tz_convert(
    RESEARCH_CONFIG["session_timezone"]
)

coverage_manifest = summarise_daily_coverage(
    all_research_timestamps,
    start_date=local_timestamps.min().date(),
    end_date=local_timestamps.max().date(),
    timezone=RESEARCH_CONFIG["session_timezone"],
)

daily_sources = pd.concat(
    source_dates,
    ignore_index=True,
).groupby("date")["dataset_id"].agg(
    lambda values: ";".join(sorted(set(values)))
)

coverage_manifest = coverage_manifest.merge(
    daily_sources.rename("source_dataset_ids"),
    left_on="date",
    right_index=True,
    how="left",
    validate="one_to_one",
)

coverage_manifest["year"] = pd.to_datetime(
    coverage_manifest["date"]
).dt.year

assert (
    coverage_manifest["observed_minutes"].sum()
    == len(all_research_timestamps)
)

print("MODEL-INDEPENDENT HISTORICAL COVERAGE")

display(
    coverage_manifest.groupby("year").agg(
        calendar_days=("date", "size"),
        observed_days=("has_observations", "sum"),
        total_candles=("observed_minutes", "sum"),
        internal_unobserved_minutes=(
            "internal_unobserved_minutes", "sum"
        ),
    )
)

print("TOTAL OBSERVATIONS")
print(len(all_research_timestamps))

In [ ]:
partition_summary = (
    coverage_manifest.loc[
        coverage_manifest["has_observations"]
    ]
    .groupby("split", sort=False)
    .agg(
        first_observed_date=("date", "min"),
        last_observed_date=("date", "max"),
        observed_days=("date", "size"),
        observations=("observed_minutes", "sum"),
    )
)

print("CHRONOLOGICAL RESEARCH PARTITIONS")

display(partition_summary)

manifest_path = (
    RESEARCH_CONFIG["research_data_root"]
    / "session_manifest.csv"
)

manifest_content = coverage_manifest.to_csv(
    index=False,
    lineterminator="\n",
).encode("utf-8")

if manifest_path.exists():

    if manifest_path.read_bytes() != manifest_content:
        raise ValueError(
            "Existing coverage manifest differs. "
            "Review the changes before replacing it."
        )

else:

    with manifest_path.open("xb") as file:
        file.write(manifest_content)

print("Coverage manifest verified.")
print("All corrected observations retained.")
print("No model-specific session exclusions applied.")

## 5. Causal Historical Reconstruction

The historical research runner reuses the existing live
engine's mathematical state-update function.

Model parameters are inherited from the existing CONFIG
rather than manually duplicated.

Anchoring, warmup and observation windows are supplied
as experimental inputs and are not assumed to be optimal.

The initial experiment reconstructs the current model
using previously examined 2025 development data.

Updated candle states are available only after the
corresponding candle has closed. They must not be used
as predictions of that same candle.

The initial reconstruction does not evaluate trading
signals or strategy profitability.

In [ ]:
from copy import deepcopy

from src.config import CONFIG
from src.research_runner import run_historical_session


# Inherit existing model parameters without modifying
# or maintaining a second copy of the live configuration.
experiment_config = deepcopy(CONFIG)

print("INHERITED MODEL CONFIGURATION")

display(
    pd.Series({
        key: experiment_config[key]
        for key in [
            "reference_type",
            "ewma_halflife",
            "ghost_lookback_bars",
            "ghost_upper_percentiles",
        ]
    }, name="value").to_frame()
)

In [ ]:
experiment_date = "2025-01-06"

# Explicit experimental inputs — not model defaults.
experiment_start_time = "07:30"
experiment_anchor_time = "09:00"
experiment_end_time = "12:30"

session_timezone = RESEARCH_CONFIG[
    "session_timezone"
]

experiment_start = pd.Timestamp(
    f"{experiment_date} {experiment_start_time}",
    tz=session_timezone,
).tz_convert("UTC")

experiment_anchor = pd.Timestamp(
    f"{experiment_date} {experiment_anchor_time}",
    tz=session_timezone,
).tz_convert("UTC")

experiment_end = pd.Timestamp(
    f"{experiment_date} {experiment_end_time}",
    tz=session_timezone,
).tz_convert("UTC")

experiment_window = pd.DataFrame({
    "start": [experiment_start],
    "end": [experiment_end],
})

window_coverage = assess_windows(
    all_research_timestamps,
    experiment_window,
)

display(window_coverage)

assert window_coverage["complete"].all(), (
    "Experimental window is incomplete."
)

baseline_record = next(
    record
    for record in raw_dataset_records
    if pd.Timestamp(
        record["requested_start"]
    ).year == 2025
)

baseline_label = baseline_record[
    "dataset_label"
].replace("_raw_", "_utc_", 1)

baseline_path = (
    VALIDATED_DATA_DIR
    / baseline_label
    / f"{baseline_label}.csv"
)

baseline_data = pd.read_csv(
    baseline_path,
    usecols=[
        "datetime",
        "open",
        "high",
        "low",
        "close",
        "tick_volume",
    ],
)

baseline_data["datetime"] = pd.to_datetime(
    baseline_data["datetime"],
    utc=True,
    errors="raise",
)

session_candles = baseline_data.loc[
    baseline_data["datetime"].ge(experiment_start)
    & baseline_data["datetime"].lt(experiment_end)
].copy()

reconstructed_session = run_historical_session(
    session_candles,
    experiment_config,
    anchor_utc=experiment_anchor,
)

del baseline_data

print(
    "Reconstructed candles:",
    len(reconstructed_session),
)

print(
    "Post-anchor candles:",
    reconstructed_session["is_post_anchor"].sum(),
)

display(reconstructed_session.head())

display(
    reconstructed_session.loc[
        reconstructed_session["is_post_anchor"]
    ].head(
        experiment_config["ghost_lookback_bars"] + 2
    )
)

In [ ]:
import matplotlib.pyplot as plt

plot_data = reconstructed_session.loc[
    reconstructed_session["is_post_anchor"]
].copy()

plot_time = plot_data["datetime"].dt.tz_convert(
    session_timezone
)

fig, ax = plt.subplots(figsize=(15, 7))

ax.plot(
    plot_time,
    plot_data["close"],
    color="black",
    linewidth=1,
    label="Close",
)

ax.plot(
    plot_time,
    plot_data["reference"],
    color="blue",
    linewidth=1.5,
    label="VWAP",
)

band_colours = {
    1: "green",
    2: "orange",
    3: "red",
}

for band, colour in band_colours.items():

    for side in ["+", "-"]:

        solid_column = f"solid_{band}{side}"
        ghost_column = f"ghost_{band}{side}"

        if solid_column in plot_data:

            ax.plot(
                plot_time,
                plot_data[solid_column],
                color=colour,
                linewidth=1,
                label=(
                    f"Solid {band}"
                    if side == "+" else None
                ),
            )

        if ghost_column in plot_data:

            ax.plot(
                plot_time,
                plot_data[ghost_column],
                color=colour,
                linestyle="--",
                alpha=0.65,
                label=(
                    f"Gaussian {band}"
                    if side == "+" else None
                ),
            )

ax.set_title(
    f"Existing Engine — Historical Reconstruction "
    f"({experiment_date})"
)

ax.set_xlabel("New York local time")
ax.set_ylabel("US100.cash price")
ax.legend()
ax.grid(alpha=0.15)

plt.tight_layout()
plt.show()

## 6. Prior and Updated Mathematical Snapshots

Predictive calibration requires comparison of the realised
candle with the mathematical state available before that
candle was observed.

The research runner preserves both:

- Prior state: estimates from the preceding completed candle.
- Updated state: estimates incorporating the current candle.

The first candle has no prior state.

Pre-anchor states are preserved but distinguished from
states belonging to the newly anchored VWAP.

Gaussian readiness is inherited from the existing engine
rather than imposed through a fixed research warmup.

No predictive calibration or model selection is performed
at this stage.

In [ ]:
snapshot_data = reconstructed_session.copy()

# Identify candles for which the existing anchored
# Gaussian model had an available prior distribution.
predictive_eligible = (
    snapshot_data["is_post_anchor"]
    & snapshot_data["prior_exists"]
    & snapshot_data["prior_is_post_anchor"]
    & snapshot_data["prior_ghost_ready"]
    & (
        snapshot_data["prior_available_at_utc"]
        <= snapshot_data["datetime"]
    )
)

print("SNAPSHOT AVAILABILITY")

display(
    pd.Series({
        "total_observations": len(snapshot_data),
        "post_anchor_observations": (
            snapshot_data["is_post_anchor"].sum()
        ),
        "updated_post_anchor_gaussian_ready": (
            snapshot_data["is_post_anchor"]
            & snapshot_data["ghost_ready"]
        ).sum(),
        "prior_post_anchor_gaussian_ready": (
            predictive_eligible.sum()
        ),
    }, name="observations").to_frame()
)

# Inspect the transition from Gaussian initialisation
# to genuine prior-state availability.
ready_positions = np.flatnonzero(
    (
        snapshot_data["is_post_anchor"]
        & snapshot_data["ghost_ready"]
    ).to_numpy()
)

if len(ready_positions):

    first_ready = ready_positions[0]

    transition = snapshot_data.iloc[
        max(0, first_ready - 2):first_ready + 4
    ].copy()

    transition["datetime_ny"] = (
        transition["datetime"]
        .dt.tz_convert(session_timezone)
    )

    display(
        transition[[
            "datetime_ny",
            "close",
            "prior_reference",
            "reference",
            "prior_ghost_sigma",
            "ghost_sigma",
            "prior_ghost_ready",
            "ghost_ready",
        ]]
    )

# Confirm that prior estimates never become available
# after the candle against which they will be evaluated.
assert (
    snapshot_data.loc[
        predictive_eligible,
        "prior_available_at_utc",
    ]
    <= snapshot_data.loc[
        predictive_eligible,
        "datetime",
    ]
).all()

## 7. Empirical Gaussian Baseline Reproduction

Reproduce the original Gaussian band calibration experiment
using the existing mathematical engine and inherited CONFIG.

Three historical data interpretations are compared:

1. Legacy historical observations and timestamp interpretation.
2. Newly acquired observations with the raw broker-clock labels.
3. Newly acquired observations with corrected UTC timestamps.

The second experiment controls for acquisition differences when
investigating the historical timestamp correction.

The original experimental window is retained exclusively for
benchmark reproduction and is not assumed optimal.

Predictive exceedances are measured against estimates available
before the realised candle.

Descriptive exceedances are measured against updated estimates
incorporating that candle.

No candidate model fitting or parameter optimisation is performed.

In [ ]:
import hashlib
import subprocess

from src.loaders import load_mt5_csv


# Inherit the current mathematical implementation.
baseline_config = deepcopy(CONFIG)

# Reuse the existing experimental time settings.
baseline_start_time = experiment_start_time
baseline_anchor_time = experiment_anchor_time
baseline_end_time = experiment_end_time

# Original benchmark evaluation window begins at 09:30.
baseline_evaluation_time = "09:30"

baseline_year = 2025

legacy_path = (
    DATA_DIR / RESEARCH_CONFIG["baseline_dataset"]
)

legacy_data = load_mt5_csv(
    str(legacy_path)
).drop_duplicates(
    subset="datetime"
)

corrected_data = pd.read_csv(
    baseline_path,
    usecols=[
        "datetime",
        "mt5_time_raw",
        "open",
        "high",
        "low",
        "close",
        "tick_volume",
    ],
)

corrected_data["datetime"] = pd.to_datetime(
    corrected_data["datetime"],
    utc=True,
    errors="raise",
)

# Same acquired observations, but with the previous
# incorrect clock interpretation.
raw_clock_data = corrected_data.copy()

raw_clock_data["datetime"] = pd.to_datetime(
    raw_clock_data["mt5_time_raw"],
    unit="s",
    utc=True,
    errors="raise",
)

baseline_sources = {
    "legacy": legacy_data,
    "raw_clock_control": raw_clock_data,
    "corrected": corrected_data,
}

# Record the engine version used in this experiment.
baseline_engine_commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(PROJECT_ROOT),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

dirty_source = subprocess.check_output(
    [
        "git",
        "-C",
        str(PROJECT_ROOT),
        "status",
        "--porcelain",
        "--",
        "src",
    ],
    text=True,
).strip()

if dirty_source:
    raise ValueError(
        "Commit existing source changes before "
        "conducting the baseline experiment."
    )

baseline_provenance = {
    "engine_commit": baseline_engine_commit,
    "model_config": deepcopy(baseline_config),
    "year": baseline_year,
    "session_timezone": session_timezone,
    "context_start": baseline_start_time,
    "anchor": baseline_anchor_time,
    "evaluation_start": baseline_evaluation_time,
    "evaluation_end_exclusive": baseline_end_time,
    "legacy_sha256": sha256_file(legacy_path),
    "corrected_sha256": sha256_file(baseline_path),
    "python_version": sys.version,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "research_role": "known_development",
}

print("BASELINE EXPERIMENT CONFIGURATION")

display(
    pd.Series({
        "reference_type": baseline_config["reference_type"],
        "ewma_halflife": baseline_config["ewma_halflife"],
        "ghost_lookback_bars": baseline_config[
            "ghost_lookback_bars"
        ],
        "ghost_upper_percentiles": baseline_config[
            "ghost_upper_percentiles"
        ],
        "context_start": baseline_start_time,
        "anchor": baseline_anchor_time,
        "evaluation_start": baseline_evaluation_time,
        "evaluation_end": baseline_end_time,
    }, name="value").to_frame()
)

print("Engine commit:", baseline_engine_commit)

display(
    pd.DataFrame([
        {
            "dataset": name,
            "observations": len(frame),
        }
        for name, frame in baseline_sources.items()
    ])
)

In [ ]:
baseline_results = {}
baseline_coverage_records = []

for dataset_name, candles in baseline_sources.items():

    timestamps = validate_timestamp_index(
        candles["datetime"]
    )

    dates = timestamps.tz_convert(
        session_timezone
    ).strftime("%Y-%m-%d")

    candidate_dates = sorted(
        date
        for date in dates.unique()
        if pd.Timestamp(date).year == baseline_year
    )

    # Construct the original experimental windows.
    windows = pd.DataFrame([
        {
            "session_date": day,
            "start": pd.Timestamp(
                f"{day} {baseline_start_time}",
                tz=session_timezone,
            ).tz_convert("UTC"),
            "anchor": pd.Timestamp(
                f"{day} {baseline_anchor_time}",
                tz=session_timezone,
            ).tz_convert("UTC"),
            "evaluation_start": pd.Timestamp(
                f"{day} {baseline_evaluation_time}",
                tz=session_timezone,
            ).tz_convert("UTC"),
            "end": pd.Timestamp(
                f"{day} {baseline_end_time}",
                tz=session_timezone,
            ).tz_convert("UTC"),
        }
        for day in candidate_dates
    ])

    coverage = assess_windows(
        timestamps,
        windows,
    )

    coverage["dataset"] = dataset_name

    baseline_coverage_records.append(coverage)

    eligible_windows = coverage.loc[
        coverage["complete"]
    ]

    indexed_candles = candles.set_index(
        "datetime",
        drop=False,
    )

    session_results = []

    for window in eligible_windows.itertuples(
        index=False
    ):

        session_candles = indexed_candles.loc[
            window.start:
            window.end - pd.Timedelta(minutes=1)
        ]

        reconstructed = run_historical_session(
            session_candles,
            baseline_config,
            anchor_utc=window.anchor,
        )

        evaluation = reconstructed.loc[
            reconstructed["datetime"].ge(
                window.evaluation_start
            )
            & reconstructed["datetime"].lt(
                window.end
            )
        ].copy()

        expected_observations = int(
            (
                window.end
                - window.evaluation_start
            ) / pd.Timedelta(minutes=1)
        )

        assert len(evaluation) == expected_observations

        evaluation["session_date"] = (
            window.session_date
        )

        session_results.append(evaluation)

    if not session_results:
        raise ValueError(
            f"No complete sessions: {dataset_name}"
        )

    baseline_results[dataset_name] = pd.concat(
        session_results,
        ignore_index=True,
    )

    print(
        f"{dataset_name}: "
        f"{len(session_results)} complete sessions, "
        f"{len(baseline_results[dataset_name]):,} "
        f"evaluation candles"
    )

baseline_coverage = pd.concat(
    baseline_coverage_records,
    ignore_index=True,
)

print("\nBASELINE SESSION COVERAGE")

display(
    baseline_coverage.groupby("dataset").agg(
        candidate_dates=("session_date", "size"),
        complete_sessions=("complete", "sum"),
        excluded_dates=(
            "complete",
            lambda x: (~x).sum(),
        ),
    )
)

In [ ]:
calibration_records = []
availability_records = []

percentiles = baseline_config[
    "ghost_upper_percentiles"
]

ghost_columns = [
    f"ghost_{level}{side}"
    for level in range(1, len(percentiles) + 1)
    for side in ("+", "-")
]

prior_ghost_columns = [
    f"prior_{column}"
    for column in ghost_columns
]

for dataset_name, observations in baseline_results.items():

    # Predictive state must exist before the observation.
    prior_mask = (
        observations["is_post_anchor"]
        & observations["prior_exists"]
        & observations["prior_is_post_anchor"]
        & observations["prior_ghost_ready"]
        & observations[prior_ghost_columns]
        .notna().all(axis=1)
        & (
            observations["prior_available_at_utc"]
            <= observations["datetime"]
        )
    )

    updated_mask = (
        observations["is_post_anchor"]
        & observations["ghost_ready"]
        & observations[ghost_columns]
        .notna().all(axis=1)
    )

    # Use identical observations when comparing predictive
    # and descriptive coverage.
    common_mask = prior_mask & updated_mask

    availability_records.append({
        "dataset": dataset_name,
        "evaluation_candles": len(observations),
        "prior_available": int(prior_mask.sum()),
        "updated_available": int(updated_mask.sum()),
        "common_available": int(common_mask.sum()),
    })

    comparisons = [
        ("predictive", "prior_", prior_mask),
        ("predictive_matched", "prior_", common_mask),
        ("descriptive_matched", "", common_mask),
    ]

    for evaluation_type, prefix, mask in comparisons:

        sample = observations.loc[mask]

        if sample.empty:
            raise ValueError(
                f"No eligible observations: "
                f"{dataset_name}, {evaluation_type}"
            )

        for level, percentile in enumerate(
            percentiles,
            start=1,
        ):

            nominal = 1.0 - percentile

            upper = sample[
                f"{prefix}ghost_{level}+"
            ]

            lower = sample[
                f"{prefix}ghost_{level}-"
            ]

            assert upper.gt(lower).all()

            upper_hits = sample["close"].gt(upper)
            lower_hits = sample["close"].lt(lower)

            for tail, hits in [
                ("upper", upper_hits),
                ("lower", lower_hits),
            ]:

                observed = float(hits.mean())

                calibration_records.append({
                    "dataset": dataset_name,
                    "evaluation": evaluation_type,
                    "band": level,
                    "tail": tail,
                    "nominal": nominal,
                    "exceedances": int(hits.sum()),
                    "observations": len(sample),
                    "sessions": sample[
                        "session_date"
                    ].nunique(),
                    "observed": observed,
                    "calibration_error": (
                        observed - nominal
                    ),
                    "exceedance_ratio": (
                        observed / nominal
                    ),
                    "mean_width": float(
                        (upper - lower).mean()
                    ),
                })

baseline_calibration = pd.DataFrame(
    calibration_records
)

baseline_availability = pd.DataFrame(
    availability_records
)

print("ESTIMATOR AVAILABILITY")
display(baseline_availability)

print("PREDICTIVE CALIBRATION")

predictive_calibration = baseline_calibration.loc[
    baseline_calibration["evaluation"].eq(
        "predictive"
    )
].copy()

display(
    predictive_calibration.style.format({
        "nominal": "{:.2%}",
        "observed": "{:.2%}",
        "calibration_error": "{:+.2%}",
        "exceedance_ratio": "{:.3f}",
        "mean_width": "{:.2f}",
    })
)

In [ ]:
from matplotlib.ticker import PercentFormatter


# Archived counts from the original research notebook.
# These are reference results, not model parameters.
archived_exceedances = {
    (1, "upper"): 11164,
    (1, "lower"): 9322,
    (2, "upper"): 3191,
    (2, "lower"): 3288,
    (3, "upper"): 396,
    (3, "lower"): 578,
}

legacy_reproduction = predictive_calibration.loc[
    predictive_calibration["dataset"].eq("legacy")
].copy()

legacy_reproduction["archived_exceedances"] = [
    archived_exceedances.get((band, tail), np.nan)
    for band, tail in zip(
        legacy_reproduction["band"],
        legacy_reproduction["tail"],
    )
]

legacy_reproduction["difference"] = (
    legacy_reproduction["exceedances"]
    - legacy_reproduction["archived_exceedances"]
)

print("ORIGINAL BASELINE REPRODUCTION")

display(
    legacy_reproduction[[
        "band",
        "tail",
        "observations",
        "exceedances",
        "archived_exceedances",
        "difference",
    ]]
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5),
    sharex=True,
    sharey=True,
)

limit = 1.1 * max(
    predictive_calibration["nominal"].max(),
    predictive_calibration["observed"].max(),
)

markers = {
    "legacy": "o",
    "raw_clock_control": "s",
    "corrected": "^",
}

for ax, tail in zip(
    axes,
    ["lower", "upper"],
):

    ax.plot(
        [0, limit],
        [0, limit],
        color="grey",
        linestyle="--",
        label="Perfect calibration",
    )

    for dataset_name, marker in markers.items():

        rows = predictive_calibration.loc[
            predictive_calibration["dataset"].eq(
                dataset_name
            )
            & predictive_calibration["tail"].eq(tail)
        ].sort_values("nominal")

        ax.scatter(
            rows["nominal"],
            rows["observed"],
            marker=marker,
            s=65,
            label=dataset_name,
        )

    ax.set_title(f"{tail.capitalize()} tail")
    ax.set_xlabel("Nominal exceedance probability")

    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)
    ax.set_aspect("equal")

    ax.xaxis.set_major_formatter(
        PercentFormatter(1)
    )

    ax.yaxis.set_major_formatter(
        PercentFormatter(1)
    )

    ax.grid(alpha=0.2)

axes[0].set_ylabel(
    "Empirical exceedance probability"
)

axes[1].legend()

fig.suptitle(
    "Gaussian Baseline — Predictive Tail Calibration"
)

plt.tight_layout()
plt.show()

In [ ]:
# Freeze the effective configuration, data identities
# and numerical results for reproducibility.

baseline_specification = json.dumps(
    baseline_provenance,
    sort_keys=True,
    default=str,
    allow_nan=False,
)

baseline_id = hashlib.sha256(
    baseline_specification.encode("utf-8")
).hexdigest()[:16]

baseline_report = {
    "experiment_id": baseline_id,
    "provenance": baseline_provenance,
    "availability": json.loads(
        baseline_availability.to_json(
            orient="records"
        )
    ),
    "calibration": json.loads(
        baseline_calibration.to_json(
            orient="records",
            double_precision=15,
        )
    ),
}

baseline_report_content = json.dumps(
    baseline_report,
    indent=2,
    sort_keys=True,
    default=str,
    allow_nan=False,
)

baseline_report_path = (
    RESEARCH_CONFIG["research_data_root"]
    / f"gaussian_baseline_{baseline_id}.json"
)

if baseline_report_path.exists():

    if baseline_report_path.read_text(
        encoding="utf-8"
    ) != baseline_report_content:

        raise ValueError(
            "Existing benchmark results differ "
            "for an identical experiment specification."
        )

else:

    with baseline_report_path.open(
        "x",
        encoding="utf-8",
    ) as file:

        file.write(baseline_report_content)

print("Benchmark experiment:", baseline_id)
print("Benchmark report:", baseline_report_path)

print("MATCHED PREDICTIVE VS DESCRIPTIVE CALIBRATION")

display(
    baseline_calibration.loc[
        baseline_calibration["dataset"].eq("corrected")
        & baseline_calibration["evaluation"].isin([
            "predictive_matched",
            "descriptive_matched",
        ])
    ].pivot(
        index=["band", "tail"],
        columns="evaluation",
        values="observed",
    ).style.format("{:.2%}")
)

## 8. Empirical Distribution and Conditional Calibration

Investigate the standardised predictive errors of the existing
Gaussian VWAP model using corrected 2025 development data.

The analysis uses only mathematical states available before
the realised candle.

The theoretical reference distribution is standard normal,
without fitting its location or scale to the observed sample.

Examine empirical moments, quantiles, density, probability
integral transforms, serial dependence and conditional
calibration.

Serial dependence is measured within sessions only.

The purpose is to diagnose model assumptions rather than
select a replacement distribution or optimise parameters.

All findings are exploratory development-sample evidence.

In [ ]:
from scipy import stats


distribution_data = baseline_results["corrected"].copy()

# Preserve the predictive eligibility criteria used
# in the baseline calibration experiment.
distribution_mask = (
    distribution_data["is_post_anchor"]
    & distribution_data["prior_exists"]
    & distribution_data["prior_is_post_anchor"]
    & distribution_data["prior_ghost_ready"]
    & distribution_data[prior_ghost_columns]
    .notna().all(axis=1)
    & (
        distribution_data["prior_available_at_utc"]
        <= distribution_data["datetime"]
    )
)

distribution_data = (
    distribution_data.loc[distribution_mask]
    .sort_values(["session_date", "datetime"])
    .reset_index(drop=True)
)

assert not distribution_data.empty

assert np.isfinite(
    distribution_data["prior_ghost_sigma"]
).all()

assert distribution_data["prior_ghost_sigma"].gt(0).all()

# Signed predictive gap.
distribution_data["gap"] = (
    distribution_data["close"]
    - distribution_data["prior_reference"]
)

# Standardised predictive error.
distribution_data["z"] = (
    distribution_data["gap"]
    / distribution_data["prior_ghost_sigma"]
)

distribution_data["abs_z"] = (
    distribution_data["z"].abs()
)

distribution_data["z_squared"] = (
    distribution_data["z"].pow(2)
)

assert np.isfinite(distribution_data["z"]).all()

# Probability integral transform under the
# existing conditional Gaussian hypothesis.
distribution_data["pit"] = stats.norm.cdf(
    distribution_data["z"]
)

z = distribution_data["z"].to_numpy()

# Verify that this is the same predictive sample
# used in the corrected baseline experiment.
expected_n = predictive_calibration.loc[
    predictive_calibration["dataset"].eq("corrected"),
    "observations",
].unique()

assert len(expected_n) == 1
assert len(z) == expected_n[0]

distribution_moments = pd.DataFrame({
    "theoretical_normal": {
        "mean": 0.0,
        "median": 0.0,
        "standard_deviation": 1.0,
        "variance": 1.0,
        "rms": 1.0,
        "skewness": 0.0,
        "excess_kurtosis": 0.0,
        "mean_absolute_value": np.sqrt(2 / np.pi),
    },
    "empirical": {
        "mean": np.mean(z),
        "median": np.median(z),
        "standard_deviation": np.std(z, ddof=1),
        "variance": np.var(z, ddof=1),
        "rms": np.sqrt(np.mean(z**2)),
        "skewness": stats.skew(z, bias=False),
        "excess_kurtosis": stats.kurtosis(
            z,
            fisher=True,
            bias=False,
        ),
        "mean_absolute_value": np.mean(np.abs(z)),
    },
})

distribution_moments["difference"] = (
    distribution_moments["empirical"]
    - distribution_moments["theoretical_normal"]
)

print("STANDARDISED PREDICTIVE ERRORS")
print(f"Observations: {len(z):,}")
print(
    "Sessions:",
    distribution_data["session_date"].nunique(),
)

display(distribution_moments.round(5))

# Inherit the quantile levels from the existing model.
upper_percentiles = np.asarray(
    baseline_config["ghost_upper_percentiles"]
)

quantile_levels = np.sort(
    np.concatenate([
        1 - upper_percentiles,
        [0.50],
        upper_percentiles,
    ])
)

empirical_quantiles = pd.DataFrame({
    "probability": quantile_levels,
    "gaussian_quantile": stats.norm.ppf(
        quantile_levels
    ),
    "empirical_quantile": np.quantile(
        z,
        quantile_levels,
    ),
})

empirical_quantiles["difference"] = (
    empirical_quantiles["empirical_quantile"]
    - empirical_quantiles["gaussian_quantile"]
)

print("EMPIRICAL VS GAUSSIAN QUANTILES")

display(
    empirical_quantiles.style.format({
        "probability": "{:.2%}",
        "gaussian_quantile": "{:.5f}",
        "empirical_quantile": "{:.5f}",
        "difference": "{:+.5f}",
    })
)

In [ ]:
distribution_fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 5),
)

# 1. Empirical density against N(0, 1).
x = np.linspace(
    min(z.min(), -4.0),
    max(z.max(), 4.0),
    1000,
)

axes[0].hist(
    z,
    bins=100,
    density=True,
    alpha=0.65,
    label="Empirical density",
)

axes[0].plot(
    x,
    stats.norm.pdf(x),
    color="black",
    linewidth=2,
    label="Standard normal",
)

axes[0].axvline(
    0,
    color="grey",
    linestyle="--",
    alpha=0.6,
)

axes[0].set_title("Standardised Error Distribution")
axes[0].set_xlabel("Standardised error")
axes[0].set_ylabel("Density")
axes[0].legend()

# 2. Gaussian Q-Q plot.
ordered_z = np.sort(z)

plotting_probabilities = (
    np.arange(1, len(z) + 1) - 0.5
) / len(z)

theoretical_quantiles = stats.norm.ppf(
    plotting_probabilities
)

axes[1].scatter(
    theoretical_quantiles,
    ordered_z,
    s=5,
    alpha=0.35,
    rasterized=True,
)

qq_min = min(
    theoretical_quantiles.min(),
    ordered_z.min(),
)

qq_max = max(
    theoretical_quantiles.max(),
    ordered_z.max(),
)

axes[1].plot(
    [qq_min, qq_max],
    [qq_min, qq_max],
    color="black",
    linestyle="--",
    label="Standard normal reference",
)

axes[1].set_title("Gaussian Q-Q Plot")
axes[1].set_xlabel("Theoretical quantiles")
axes[1].set_ylabel("Empirical quantiles")
axes[1].legend()

# 3. Probability integral transform.
pit = distribution_data["pit"].to_numpy()

pit_edges = np.linspace(0, 1, 11)

pit_counts, _ = np.histogram(
    pit,
    bins=pit_edges,
)

pit_frequencies = pit_counts / len(pit)

axes[2].bar(
    pit_edges[:-1],
    pit_frequencies,
    width=np.diff(pit_edges),
    align="edge",
    alpha=0.7,
    edgecolor="white",
)

axes[2].axhline(
    1 / len(pit_counts),
    color="black",
    linestyle="--",
    label="Uniform reference",
)

axes[2].set_xlim(0, 1)
axes[2].set_title("Probability Integral Transform")
axes[2].set_xlabel("PIT value")
axes[2].set_ylabel("Observed probability")
axes[2].yaxis.set_major_formatter(
    PercentFormatter(1)
)
axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.15)

distribution_fig.suptitle(
    "Existing Gaussian Model — Distributional Diagnostics"
)

distribution_fig.tight_layout()
plt.show()

In [ ]:
# Diagnostic horizons in minutes, not model parameters.
diagnostic_lags = [1, 2, 5, 10, 20, 30, 60]

dependence_records = []

session_groups = distribution_data.groupby(
    "session_date",
    sort=False,
)

for lag in diagnostic_lags:

    lagged_time = session_groups[
        "datetime"
    ].shift(lag)

    # Never correlate observations across session
    # boundaries or across missing minute intervals.
    valid_pairs = (
        distribution_data["datetime"] - lagged_time
    ).eq(
        pd.Timedelta(minutes=lag)
    )

    for column in [
        "gap",
        "z",
        "abs_z",
        "z_squared",
        "pit",
    ]:

        lagged_values = session_groups[
            column
        ].shift(lag)

        current = distribution_data.loc[
            valid_pairs,
            column,
        ]

        previous = lagged_values.loc[
            valid_pairs
        ]

        dependence_records.append({
            "variable": column,
            "lag_minutes": lag,
            "observation_pairs": int(
                valid_pairs.sum()
            ),
            "correlation": current.corr(previous),
        })

dependence_table = pd.DataFrame(
    dependence_records
)

print("WITHIN-SESSION LAGGED CORRELATIONS")

display(
    dependence_table.pivot(
        index="lag_minutes",
        columns="variable",
        values="correlation",
    ).round(4)
)

dependence_fig, ax = plt.subplots(
    figsize=(12, 6)
)

for column in [
    "gap",
    "z",
    "abs_z",
    "z_squared",
    "pit",
]:

    rows = dependence_table.loc[
        dependence_table["variable"].eq(column)
    ]

    ax.plot(
        rows["lag_minutes"],
        rows["correlation"],
        marker="o",
        label=column,
    )

ax.axhline(
    0,
    color="black",
    linestyle="--",
    alpha=0.6,
)

ax.set_title(
    "Persistence of Gaussian Forecast Errors"
)

ax.set_xlabel("Lag (minutes)")
ax.set_ylabel("Pooled within-session correlation")
ax.set_ylim(-1.05, 1.05)
ax.set_xticks(diagnostic_lags)
ax.legend()
ax.grid(alpha=0.2)

dependence_fig.tight_layout()
plt.show()

In [ ]:
# Thirty-minute diagnostic groups.
# These do not modify the estimator.
distribution_data["ny_time_block"] = (
    distribution_data["datetime"]
    .dt.tz_convert(session_timezone)
    .dt.floor("30min")
    .dt.strftime("%H:%M")
)

# Direction known before the current candle.
prior_displacement = (
    distribution_data["prior_close"]
    - distribution_data["prior_reference"]
)

distribution_data["prior_direction"] = np.select(
    [
        prior_displacement.gt(0),
        prior_displacement.lt(0),
    ],
    [
        "above_vwap",
        "below_vwap",
    ],
    default="at_vwap",
)

# Calculate actual predictive exceedances against
# the inherited model's prior band boundaries.
tail_aggregations = {}

for level, percentile in enumerate(
    baseline_config["ghost_upper_percentiles"],
    start=1,
):

    upper_column = f"upper_hit_{level}"
    lower_column = f"lower_hit_{level}"

    distribution_data[upper_column] = (
        distribution_data["close"]
        > distribution_data[
            f"prior_ghost_{level}+"
        ]
    )

    distribution_data[lower_column] = (
        distribution_data["close"]
        < distribution_data[
            f"prior_ghost_{level}-"
        ]
    )

    tail_aggregations[
        f"upper_tail_{level}"
    ] = (upper_column, "mean")

    tail_aggregations[
        f"lower_tail_{level}"
    ] = (lower_column, "mean")

diagnostic_aggregations = {
    "observations": ("z", "size"),
    "sessions": ("session_date", "nunique"),
    "mean_z": ("z", "mean"),
    "std_z": ("z", "std"),
    "mean_z_squared": ("z_squared", "mean"),
    "mean_prior_scale": (
        "prior_ghost_sigma",
        "mean",
    ),
    **tail_aggregations,
}

intraday_diagnostics = (
    distribution_data.groupby(
        "ny_time_block",
        sort=True,
    ).agg(**diagnostic_aggregations)
)

directional_diagnostics = (
    distribution_data.groupby(
        "prior_direction",
        sort=True,
    ).agg(**diagnostic_aggregations)
)

tail_formats = {
    column: "{:.2%}"
    for column in tail_aggregations
}

print("INTRADAY CONDITIONAL CALIBRATION")

display(
    intraday_diagnostics.style.format({
        "mean_z": "{:+.4f}",
        "std_z": "{:.4f}",
        "mean_z_squared": "{:.4f}",
        "mean_prior_scale": "{:.2f}",
        **tail_formats,
    })
)

print("CALIBRATION CONDITIONAL ON PRIOR DIRECTION")

display(
    directional_diagnostics.style.format({
        "mean_z": "{:+.4f}",
        "std_z": "{:.4f}",
        "mean_z_squared": "{:.4f}",
        "mean_prior_scale": "{:.2f}",
        **tail_formats,
    })
)

conditional_fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
)

# Conditional location and dispersion.
axes[0].plot(
    intraday_diagnostics.index,
    intraday_diagnostics["mean_z"],
    marker="o",
    label="Mean standardised error",
)

axes[0].plot(
    intraday_diagnostics.index,
    intraday_diagnostics["std_z"],
    marker="s",
    label="Standard deviation",
)

axes[0].axhline(
    0,
    color="grey",
    linestyle="--",
    alpha=0.6,
)

axes[0].axhline(
    1,
    color="grey",
    linestyle=":",
    alpha=0.6,
)

axes[0].set_title("Intraday Location and Dispersion")
axes[0].set_xlabel("New York time")
axes[0].set_ylabel("Standardised statistic")
axes[0].legend()

# Green-band predictive exceedances.
nominal_green_tail = (
    1 - baseline_config[
        "ghost_upper_percentiles"
    ][0]
)

axes[1].plot(
    intraday_diagnostics.index,
    intraday_diagnostics["upper_tail_1"],
    marker="o",
    label="Upper tail",
)

axes[1].plot(
    intraday_diagnostics.index,
    intraday_diagnostics["lower_tail_1"],
    marker="s",
    label="Lower tail",
)

axes[1].axhline(
    nominal_green_tail,
    color="black",
    linestyle="--",
    label="Nominal probability",
)

axes[1].set_title("Intraday Green-Band Calibration")
axes[1].set_xlabel("New York time")
axes[1].set_ylabel("Exceedance probability")

axes[1].yaxis.set_major_formatter(
    PercentFormatter(1)
)

axes[1].legend()

for ax in axes:
    ax.grid(alpha=0.2)

conditional_fig.tight_layout()
plt.show()

## 9. Naive Return-Based Predictive Comparator

Compare the existing Gaussian VWAP model against a
zero-drift Gaussian random-walk benchmark.

The naive model centres its predictive distribution on
the previous completed candle's closing price.

Conditional scale is estimated using the rolling root
mean square of one-minute price changes.

The lookback and probability levels are inherited from
the existing Gaussian model to avoid additional
parameter selection.

Both estimators use only completed historical candles
and are evaluated on identical eligible observations.

The comparison uses corrected 2025 development data.
No validation or holdout data is used for model selection.

The purpose is to establish a simple predictive reference,
not to optimise or develop a trading strategy.

In [ ]:
comparator_lookback = int(
    baseline_config["ghost_lookback_bars"]
)

assert comparator_lookback > 0

# Reuse the complete experimental sessions from
# the corrected Gaussian baseline.
comparator_windows = baseline_coverage.loc[
    baseline_coverage["dataset"].eq("corrected")
    & baseline_coverage["complete"]
]

comparator_candles = corrected_data.set_index(
    "datetime"
).sort_index()

naive_sessions = []

for window in comparator_windows.itertuples(
    index=False
):

    # Begin at the same anchor as the Gaussian
    # gap estimator. Do not use overnight returns
    # or observations from preceding sessions.
    prices = comparator_candles.loc[
        window.anchor:
        window.end - pd.Timedelta(minutes=1),
        "close",
    ].copy()

    # One-minute arithmetic price changes.
    price_changes = prices.diff()

    # Rolling RMS calculated after each candle.
    updated_return_scale = np.sqrt(
        price_changes.pow(2)
        .rolling(
            window=comparator_lookback,
            min_periods=comparator_lookback,
        )
        .mean()
    )

    # Shift by one candle: only previously
    # available observations enter the forecast.
    prior_return_scale = (
        updated_return_scale.shift(1)
    )

    prior_timestamp = (
        prices.index.to_series().shift(1)
    )

    naive_session = pd.DataFrame({
        "naive_prior_close": prices.shift(1),
        "naive_prior_sigma": prior_return_scale,
        "naive_available_at_utc": (
            prior_timestamp
            + pd.Timedelta(minutes=1)
        ),
    })

    naive_session = naive_session.loc[
        naive_session.index.ge(
            window.evaluation_start
        )
        if hasattr(naive_session.index, "ge")
        else naive_session.index
        >= window.evaluation_start
    ].copy()

    naive_session["session_date"] = (
        window.session_date
    )

    naive_sessions.append(
        naive_session.reset_index()
    )

naive_forecasts = pd.concat(
    naive_sessions,
    ignore_index=True,
)

assert not naive_forecasts.duplicated(
    subset=["session_date", "datetime"]
).any()

print("NAIVE PREDICTIVE FORECASTS")
print("Inherited lookback:", comparator_lookback)
print("Observations:", len(naive_forecasts))

display(naive_forecasts.head())

In [ ]:
comparison_all = distribution_data.merge(
    naive_forecasts,
    on=["session_date", "datetime"],
    how="left",
    validate="one_to_one",
)

assert len(comparison_all) == len(
    distribution_data
)

naive_ready = (
    comparison_all["naive_prior_close"].notna()
    & np.isfinite(
        comparison_all["naive_prior_sigma"]
    )
    & comparison_all["naive_prior_sigma"].gt(0)
    & (
        comparison_all["naive_available_at_utc"]
        <= comparison_all["datetime"]
    )
)

comparison_data = (
    comparison_all.loc[naive_ready]
    .sort_values(["session_date", "datetime"])
    .reset_index(drop=True)
)

assert not comparison_data.empty

# Verify that the naive forecast is centred on
# exactly the same previous close recorded by
# the existing historical research engine.
np.testing.assert_allclose(
    comparison_data["naive_prior_close"],
    comparison_data["prior_close"],
    rtol=0,
    atol=1e-9,
)

comparison_data["naive_error"] = (
    comparison_data["close"]
    - comparison_data["naive_prior_close"]
)

comparison_data["naive_z"] = (
    comparison_data["naive_error"]
    / comparison_data["naive_prior_sigma"]
)

comparison_data["naive_pit"] = stats.norm.cdf(
    comparison_data["naive_z"]
)

assert np.isfinite(
    comparison_data["naive_z"]
).all()

comparison_availability = pd.DataFrame({
    "observations": {
        "vwap_predictive_sample": len(
            distribution_data
        ),
        "matched_comparison_sample": len(
            comparison_data
        ),
        "excluded_naive_unavailable": (
            len(distribution_data)
            - len(comparison_data)
        ),
    }
})

print("MATCHED PREDICTIVE SAMPLE")

display(comparison_availability)

print(
    "Matched sessions:",
    comparison_data["session_date"].nunique(),
)

display(
    comparison_data[[
        "datetime",
        "close",
        "prior_reference",
        "prior_ghost_sigma",
        "naive_prior_close",
        "naive_prior_sigma",
        "z",
        "naive_z",
    ]].head()
)

In [ ]:
comparison_models = {
    "VWAP Gaussian": (
        "prior_reference",
        "prior_ghost_sigma",
    ),
    "Naive random walk": (
        "naive_prior_close",
        "naive_prior_sigma",
    ),
}

comparison_moment_records = []
comparison_tail_records = []

for model_name, (
    centre_column,
    scale_column,
) in comparison_models.items():

    centre = comparison_data[centre_column]
    scale = comparison_data[scale_column]

    error = comparison_data["close"] - centre

    standardised_error = error / scale

    comparison_moment_records.append({
        "model": model_name,
        "observations": len(error),
        "mean_error_points": error.mean(),
        "mae_points": error.abs().mean(),
        "rmse_points": np.sqrt(
            error.pow(2).mean()
        ),
        "mean_predictive_scale": scale.mean(),
        "mean_z": standardised_error.mean(),
        "std_z": standardised_error.std(ddof=1),
        "skewness": stats.skew(
            standardised_error,
            bias=False,
        ),
        "excess_kurtosis": stats.kurtosis(
            standardised_error,
            fisher=True,
            bias=False,
        ),
    })

    for level, percentile in enumerate(
        baseline_config["ghost_upper_percentiles"],
        start=1,
    ):

        nominal = 1.0 - percentile

        if model_name == "VWAP Gaussian":

            # Preserve the actual existing
            # engine's predictive band values.
            upper = comparison_data[
                f"prior_ghost_{level}+"
            ]

            lower = comparison_data[
                f"prior_ghost_{level}-"
            ]

        else:

            # Gaussian quantiles centred on
            # the previous closing price.
            upper = (
                centre
                + scale * stats.norm.ppf(
                    percentile
                )
            )

            lower = (
                centre
                + scale * stats.norm.ppf(
                    1 - percentile
                )
            )

        upper_hits = comparison_data[
            "close"
        ].gt(upper)

        lower_hits = comparison_data[
            "close"
        ].lt(lower)

        central_coverage = (
            ~(upper_hits | lower_hits)
        ).mean()

        comparison_tail_records.append({
            "model": model_name,
            "band": level,
            "observations": len(comparison_data),
            "nominal_tail": nominal,
            "upper_exceedances": int(
                upper_hits.sum()
            ),
            "lower_exceedances": int(
                lower_hits.sum()
            ),
            "upper_observed": upper_hits.mean(),
            "lower_observed": lower_hits.mean(),
            "nominal_coverage": (
                2 * percentile - 1
            ),
            "observed_coverage": central_coverage,
            "mean_width_points": (
                upper - lower
            ).mean(),
        })

comparison_summary = pd.DataFrame(
    comparison_moment_records
).set_index("model")

comparison_calibration = pd.DataFrame(
    comparison_tail_records
)

print("PREDICTIVE LOCATION AND DISTRIBUTION")

display(comparison_summary.T.round(5))

print("MATCHED PROBABILITY CALIBRATION")

display(
    comparison_calibration.style.format({
        "nominal_tail": "{:.2%}",
        "upper_observed": "{:.2%}",
        "lower_observed": "{:.2%}",
        "nominal_coverage": "{:.2%}",
        "observed_coverage": "{:.2%}",
        "mean_width_points": "{:.2f}",
    })
)

In [ ]:
comparison_fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 5),
)

# 1. Predictive error density.
vwap_z = comparison_data["z"].to_numpy()
naive_z = comparison_data["naive_z"].to_numpy()

density_bins = np.linspace(
    min(vwap_z.min(), naive_z.min()),
    max(vwap_z.max(), naive_z.max()),
    121,
)

axes[0].hist(
    vwap_z,
    bins=density_bins,
    density=True,
    alpha=0.50,
    label="VWAP Gaussian",
)

axes[0].hist(
    naive_z,
    bins=density_bins,
    density=True,
    alpha=0.50,
    label="Naive random walk",
)

x_grid = np.linspace(
    density_bins[0],
    density_bins[-1],
    1000,
)

axes[0].plot(
    x_grid,
    stats.norm.pdf(x_grid),
    color="black",
    linestyle="--",
    linewidth=1.8,
    label="Standard normal",
)

axes[0].set_title(
    "Standardised Predictive Errors"
)

axes[0].set_xlabel("Standardised error")
axes[0].set_ylabel("Density")
axes[0].legend()

# 2. Probability integral transform.
for column, model_name in [
    ("pit", "VWAP Gaussian"),
    ("naive_pit", "Naive random walk"),
]:

    counts, edges = np.histogram(
        comparison_data[column],
        bins=pit_edges,
    )

    axes[1].stairs(
        counts / len(comparison_data),
        edges,
        linewidth=2,
        label=model_name,
    )

axes[1].axhline(
    1 / (len(pit_edges) - 1),
    color="black",
    linestyle="--",
    label="Uniform reference",
)

axes[1].set_title(
    "Probability Integral Transform"
)

axes[1].set_xlabel("PIT value")
axes[1].set_ylabel("Observed probability")
axes[1].yaxis.set_major_formatter(
    PercentFormatter(1)
)
axes[1].legend()

# 3. Central interval calibration.
for model_name in comparison_models:

    rows = comparison_calibration.loc[
        comparison_calibration["model"].eq(
            model_name
        )
    ].sort_values("nominal_coverage")

    axes[2].plot(
        rows["nominal_coverage"],
        rows["observed_coverage"],
        marker="o",
        linewidth=1.8,
        label=model_name,
    )

axes[2].plot(
    [0, 1],
    [0, 1],
    color="black",
    linestyle="--",
    label="Perfect calibration",
)

axes[2].set_xlim(0, 1.01)
axes[2].set_ylim(0, 1.01)
axes[2].set_aspect("equal")

axes[2].set_title(
    "Predictive Interval Calibration"
)

axes[2].set_xlabel("Nominal coverage")
axes[2].set_ylabel("Empirical coverage")

axes[2].xaxis.set_major_formatter(
    PercentFormatter(1)
)

axes[2].yaxis.set_major_formatter(
    PercentFormatter(1)
)

axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.15)

comparison_fig.suptitle(
    "Gaussian VWAP vs Naive Random-Walk Benchmark"
)

comparison_fig.tight_layout()
plt.show()

## 10. Proper Scoring Rules and Statistical Uncertainty

Evaluate the Gaussian VWAP model and naive random-walk
benchmark using identical predictive observations.

Calculate Gaussian negative log-likelihood, continuous
ranked probability score and quantile pinball loss.

CRPS is the primary distributional comparison metric.

Retain the original solid RMS bands as an additional
structural reference without imposing unsupported
probability interpretations.

Estimate uncertainty using paired whole-session bootstrap
resampling to preserve within-session serial dependence.

Report confidence intervals for predictive score differences
and empirical tail exceedance probabilities.

All results remain exploratory development-sample evidence.
No estimator selection, parameter optimisation or holdout
evaluation is performed.

In [ ]:
score_frames = {}
quantile_score_records = []
tail_hit_frames = {}

score_quantiles = np.asarray(
    quantile_levels,
    dtype=float,
)

for model_name, (
    centre_column,
    scale_column,
) in comparison_models.items():

    centre = comparison_data[
        centre_column
    ].to_numpy(dtype=float)

    scale = comparison_data[
        scale_column
    ].to_numpy(dtype=float)

    realised = comparison_data[
        "close"
    ].to_numpy(dtype=float)

    assert np.isfinite(centre).all()
    assert np.isfinite(scale).all()
    assert (scale > 0).all()

    error = realised - centre
    standardised_error = error / scale

    model_scores = pd.DataFrame(
        index=comparison_data.index
    )

    model_scores["MAE"] = np.abs(error)

    # Gaussian negative log-likelihood.
    model_scores["NLL"] = (
        0.5 * np.log(2 * np.pi)
        + np.log(scale)
        + 0.5 * standardised_error**2
    )

    # Verify the analytical expression against SciPy.
    np.testing.assert_allclose(
        model_scores["NLL"],
        -stats.norm.logpdf(
            realised,
            loc=centre,
            scale=scale,
        ),
        rtol=1e-10,
        atol=1e-10,
    )

    # Analytical Gaussian CRPS.
    model_scores["CRPS"] = scale * (
        standardised_error
        * (
            2 * stats.norm.cdf(
                standardised_error
            ) - 1
        )
        + 2 * stats.norm.pdf(
            standardised_error
        )
        - 1 / np.sqrt(np.pi)
    )

    assert model_scores["CRPS"].ge(0).all()

    pinball_columns = []

    # Evaluate all inherited probability levels,
    # including the predictive median.
    for index, tau in enumerate(score_quantiles):

        predicted_quantile = (
            centre + scale * stats.norm.ppf(tau)
        )

        quantile_error = (
            realised - predicted_quantile
        )

        pinball_loss = np.maximum(
            tau * quantile_error,
            (tau - 1) * quantile_error,
        )

        column = f"pinball_{index}"

        model_scores[column] = pinball_loss

        pinball_columns.append(column)

        quantile_score_records.append({
            "model": model_name,
            "quantile": tau,
            "mean_pinball_loss": (
                pinball_loss.mean()
            ),
        })

    model_scores["Mean pinball"] = (
        model_scores[pinball_columns]
        .mean(axis=1)
    )

    assert np.isfinite(
        model_scores.to_numpy()
    ).all()

    score_frames[model_name] = model_scores

    # Preserve individual tail indicators for
    # session-bootstrap calibration intervals.
    model_tail_hits = pd.DataFrame(
        index=comparison_data.index
    )

    for level, percentile in enumerate(
        baseline_config["ghost_upper_percentiles"],
        start=1,
    ):

        upper = (
            centre
            + scale * stats.norm.ppf(percentile)
        )

        lower = (
            centre
            + scale * stats.norm.ppf(
                1 - percentile
            )
        )

        if model_name == "VWAP Gaussian":

            # Verify that Gaussian quantiles agree
            # with the actual engine boundaries.
            np.testing.assert_allclose(
                upper,
                comparison_data[
                    f"prior_ghost_{level}+"
                ],
                rtol=1e-10,
                atol=1e-8,
            )

            np.testing.assert_allclose(
                lower,
                comparison_data[
                    f"prior_ghost_{level}-"
                ],
                rtol=1e-10,
                atol=1e-8,
            )

            upper = comparison_data[
                f"prior_ghost_{level}+"
            ].to_numpy()

            lower = comparison_data[
                f"prior_ghost_{level}-"
            ].to_numpy()

        model_tail_hits[
            f"upper_{level}"
        ] = (realised > upper).astype(float)

        model_tail_hits[
            f"lower_{level}"
        ] = (realised < lower).astype(float)

    tail_hit_frames[model_name] = (
        model_tail_hits
    )


primary_score_columns = [
    "MAE",
    "NLL",
    "CRPS",
    "Mean pinball",
]

score_summary = pd.DataFrame({
    model_name: frame[
        primary_score_columns
    ].mean()
    for model_name, frame in score_frames.items()
})

score_summary["VWAP minus naive"] = (
    score_summary["VWAP Gaussian"]
    - score_summary["Naive random walk"]
)

quantile_score_summary = pd.DataFrame(
    quantile_score_records
)

print("PROPER PREDICTIVE SCORING RULES")

display(score_summary.round(6))

print("QUANTILE-SPECIFIC PINBALL LOSS")

display(
    quantile_score_summary.pivot(
        index="quantile",
        columns="model",
        values="mean_pinball_loss",
    ).style.format("{:.6f}")
)

In [ ]:
solid_baseline_records = []

solid_levels = sorted(
    int(column[len("prior_solid_"):-1])
    for column in comparison_data.columns
    if column.startswith("prior_solid_")
    and column.endswith("+")
)

assert solid_levels

for level in solid_levels:

    for evaluation, prefix in [
        ("predictive", "prior_"),
        ("descriptive", ""),
    ]:

        upper = comparison_data[
            f"{prefix}solid_{level}+"
        ]

        lower = comparison_data[
            f"{prefix}solid_{level}-"
        ]

        assert np.isfinite(upper).all()
        assert np.isfinite(lower).all()
        assert upper.ge(lower).all()

        upper_hits = comparison_data[
            "close"
        ].gt(upper)

        lower_hits = comparison_data[
            "close"
        ].lt(lower)

        solid_baseline_records.append({
            "evaluation": evaluation,
            "band": level,
            "observations": len(comparison_data),
            "sessions": comparison_data[
                "session_date"
            ].nunique(),
            "upper_exceedances": int(
                upper_hits.sum()
            ),
            "lower_exceedances": int(
                lower_hits.sum()
            ),
            "upper_observed": upper_hits.mean(),
            "lower_observed": lower_hits.mean(),
            "observed_coverage": (
                ~(upper_hits | lower_hits)
            ).mean(),
            "mean_width_points": (
                upper - lower
            ).mean(),
        })

solid_baseline_summary = pd.DataFrame(
    solid_baseline_records
)

print("ORIGINAL SOLID RMS BAND DIAGNOSTICS")

display(
    solid_baseline_summary.style.format({
        "upper_observed": "{:.2%}",
        "lower_observed": "{:.2%}",
        "observed_coverage": "{:.2%}",
        "mean_width_points": "{:.4f}",
    })
)

In [ ]:
bootstrap_replications = 2000
bootstrap_confidence = 0.95

bootstrap_rng = np.random.default_rng(
    RESEARCH_CONFIG["random_seed"]
)

session_labels = comparison_data[
    "session_date"
]

session_counts = (
    comparison_data.groupby(
        "session_date",
        sort=True,
    ).size()
)

session_ids = session_counts.index

number_of_sessions = len(session_ids)

assert number_of_sessions > 1

# Resample complete sessions with replacement.
# Reuse these draws for every model and metric.
bootstrap_draws = bootstrap_rng.integers(
    low=0,
    high=number_of_sessions,
    size=(
        bootstrap_replications,
        number_of_sessions,
    ),
)

bootstrap_observation_counts = (
    session_counts.to_numpy()[
        bootstrap_draws
    ].sum(axis=1)
)

bootstrap_alpha = (
    1 - bootstrap_confidence
)

bootstrap_quantiles = [
    bootstrap_alpha / 2,
    1 - bootstrap_alpha / 2,
]


def bootstrap_session_mean(values):

    # Aggregate by session before resampling.
    # Weight by observations rather than treating
    # unequal-length sessions equally.
    assert values.index.equals(
        comparison_data.index
    )

    assert np.isfinite(
        values.to_numpy()
    ).all()

    session_totals = (
        values.groupby(session_labels)
        .sum()
        .reindex(session_ids)
    )

    assert session_totals.notna().all()

    sampled_totals = (
        session_totals.to_numpy()[
            bootstrap_draws
        ].sum(axis=1)
    )

    return (
        sampled_totals
        / bootstrap_observation_counts
    )


# Positive differences favour the naive model.
paired_score_differences = (
    score_frames["VWAP Gaussian"]
    - score_frames["Naive random walk"]
)

bootstrap_score_records = []
bootstrap_score_samples = {}

score_metrics = (
    primary_score_columns
    + pinball_columns
)

score_metric_labels = {
    column: f"Pinball {tau:.1%}"
    for column, tau in zip(
        pinball_columns,
        score_quantiles,
    )
}

for metric in score_metrics:

    differences = paired_score_differences[
        metric
    ]

    bootstrap_means = bootstrap_session_mean(
        differences
    )

    bootstrap_score_samples[metric] = (
        bootstrap_means
    )

    lower_ci, upper_ci = np.quantile(
        bootstrap_means,
        bootstrap_quantiles,
    )

    bootstrap_score_records.append({
        "metric": metric,
        "label": score_metric_labels.get(
            metric,
            metric,
        ),
        "mean_difference": differences.mean(),
        "bootstrap_se": bootstrap_means.std(
            ddof=1
        ),
        "ci_lower": lower_ci,
        "ci_upper": upper_ci,
        "confidence_level": bootstrap_confidence,
        "replications": bootstrap_replications,
        "sessions": number_of_sessions,
        "ci_direction": (
            "favours naive"
            if lower_ci > 0
            else "favours VWAP"
            if upper_ci < 0
            else "includes zero"
        ),
    })

bootstrap_score_summary = pd.DataFrame(
    bootstrap_score_records
)

print("PAIRED SESSION-BOOTSTRAP RESULTS")
print("Sessions:", number_of_sessions)
print("Replications:", bootstrap_replications)
print("Seed:", RESEARCH_CONFIG["random_seed"])

display(
    bootstrap_score_summary.style.format({
        "mean_difference": "{:.6f}",
        "bootstrap_se": "{:.6f}",
        "ci_lower": "{:.6f}",
        "ci_upper": "{:.6f}",
        "confidence_level": "{:.0%}",
    })
)

In [ ]:
bootstrap_calibration_records = []

for model_name, hits_frame in (
    tail_hit_frames.items()
):

    for level, percentile in enumerate(
        baseline_config["ghost_upper_percentiles"],
        start=1,
    ):

        nominal_tail = 1 - percentile

        for tail in ["upper", "lower"]:

            hits = hits_frame[
                f"{tail}_{level}"
            ]

            bootstrap_rates = (
                bootstrap_session_mean(hits)
            )

            lower_ci, upper_ci = np.quantile(
                bootstrap_rates,
                bootstrap_quantiles,
            )

            observed = hits.mean()

            bootstrap_calibration_records.append({
                "model": model_name,
                "band": level,
                "tail": tail,
                "nominal": nominal_tail,
                "observed": observed,
                "calibration_error": (
                    observed - nominal_tail
                ),
                "ci_lower": lower_ci,
                "ci_upper": upper_ci,
                "nominal_inside_ci": bool(
                    lower_ci
                    <= nominal_tail
                    <= upper_ci
                ),
                "confidence_level": (
                    bootstrap_confidence
                ),
            })

bootstrap_calibration = pd.DataFrame(
    bootstrap_calibration_records
)

print("SESSION-BOOTSTRAP CALIBRATION INTERVALS")

display(
    bootstrap_calibration.style.format({
        "nominal": "{:.2%}",
        "observed": "{:.2%}",
        "calibration_error": "{:+.2%}",
        "ci_lower": "{:.2%}",
        "ci_upper": "{:.2%}",
        "confidence_level": "{:.0%}",
    })
)

In [ ]:
uncertainty_fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 6),
)

# 1. Paired quantile score differences.
quantile_bootstrap = (
    bootstrap_score_summary
    .set_index("metric")
    .loc[pinball_columns]
)

axes[0].vlines(
    score_quantiles,
    quantile_bootstrap["ci_lower"],
    quantile_bootstrap["ci_upper"],
    linewidth=2,
    color="tab:blue",
    alpha=0.8,
)

axes[0].scatter(
    score_quantiles,
    quantile_bootstrap["mean_difference"],
    color="tab:blue",
    s=45,
    zorder=3,
)

axes[0].axhline(
    0,
    color="black",
    linestyle="--",
    label="Equal predictive performance",
)

axes[0].set_title(
    "Paired Quantile Score Differences"
)

axes[0].set_xlabel("Predictive quantile")
axes[0].set_ylabel(
    "VWAP loss minus naive loss"
)

axes[0].xaxis.set_major_formatter(
    PercentFormatter(1)
)

axes[0].legend()

# 2. Session-bootstrap distribution of
# the primary comparison metric.
crps_samples = bootstrap_score_samples[
    "CRPS"
]

crps_result = bootstrap_score_summary.loc[
    bootstrap_score_summary["metric"].eq(
        "CRPS"
    )
].iloc[0]

axes[1].hist(
    crps_samples,
    bins=50,
    density=True,
    alpha=0.7,
)

axes[1].axvline(
    crps_result["mean_difference"],
    color="black",
    linewidth=2,
    label="Observed mean difference",
)

axes[1].axvline(
    crps_result["ci_lower"],
    color="grey",
    linestyle="--",
    label="95% bootstrap limits",
)

axes[1].axvline(
    crps_result["ci_upper"],
    color="grey",
    linestyle="--",
)

axes[1].set_title(
    "Bootstrap CRPS Difference"
)

axes[1].set_xlabel(
    "VWAP CRPS minus naive CRPS (points)"
)

axes[1].set_ylabel("Density")
axes[1].legend()

# 3. Tail calibration errors with uncertainty.
calibration_groups = [
    (level, tail)
    for level in range(
        1,
        len(
            baseline_config[
                "ghost_upper_percentiles"
            ]
        ) + 1,
    )
    for tail in ["upper", "lower"]
]

positions = np.arange(
    len(calibration_groups)
)

model_offsets = {
    "VWAP Gaussian": -0.12,
    "Naive random walk": 0.12,
}

model_colours = {
    "VWAP Gaussian": "tab:blue",
    "Naive random walk": "tab:green",
}

for model_name in comparison_models:

    rows = (
        bootstrap_calibration.loc[
            bootstrap_calibration["model"].eq(
                model_name
            )
        ]
        .set_index(["band", "tail"])
        .loc[calibration_groups]
    )

    adjusted_positions = (
        positions
        + model_offsets[model_name]
    )

    axes[2].vlines(
        adjusted_positions,
        rows["ci_lower"] - rows["nominal"],
        rows["ci_upper"] - rows["nominal"],
        color=model_colours[model_name],
        linewidth=2,
    )

    axes[2].scatter(
        adjusted_positions,
        rows["calibration_error"],
        color=model_colours[model_name],
        s=45,
        zorder=3,
        label=model_name,
    )

axes[2].axhline(
    0,
    color="black",
    linestyle="--",
    label="Perfect calibration",
)

axes[2].set_xticks(positions)

axes[2].set_xticklabels([
    f"{level} {tail}"
    for level, tail in calibration_groups
], rotation=30)

axes[2].set_title(
    "Tail Calibration Uncertainty"
)

axes[2].set_ylabel(
    "Observed minus nominal probability"
)

axes[2].yaxis.set_major_formatter(
    PercentFormatter(1)
)

axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.2)

uncertainty_fig.suptitle(
    "Matched Predictive Comparison — Statistical Uncertainty"
)

uncertainty_fig.tight_layout()
plt.show()